# 04 — Memory with Amazon Bedrock AgentCore

**Production question:** *Does the agent remember the customer across restarts, servers and visits?*

In production the agent runs on many short-lived instances, so in-process memory is useless. **AgentCore Memory** is a managed service with two tiers:

| Tier | Stores | Example |
|---|---|---|
| **Short-term** | The raw conversation for a session | "Continue where we left off" after a crash or redeploy |
| **Long-term** | Facts and preferences *extracted* automatically in the background | Next week, the agent already knows Kavya prefers WhatsApp |

Strands plugs into it through a **session manager**, so the agent code itself doesn't change.

```
agent turn ──► AgentCoreMemorySessionManager ──► create_event (short-term)
                                                     │  async extraction
new session ◄── retrieve preferences / facts ◄── long-term records
```

Run the carry-over cell below (collapsed). It holds the code from the earlier notebooks, so this notebook runs on its own.

In [ ]:
from strands import tool

# --- Business constants (shared with the policy layer in steering.py) ---
RETURN_WINDOW_DAYS = 30
SUPERVISOR_REVIEW_THRESHOLD = 50_000  # ₹ — refunds at or above this need a human

# --- Mock data ---
CUSTOMERS = {
    "C-1001": {"name": "Priya Sharma", "city": "Bengaluru", "email": "priya.sharma@example.com",
               "phone": "+91 98450 12345", "account_status": "active"},
    "C-1002": {"name": "Arjun Reddy", "city": "Hyderabad", "email": "arjun.reddy@example.com",
               "phone": "+91 99490 67890", "account_status": "active"},
    "C-1003": {"name": "Kavya Iyer", "city": "Chennai", "email": "kavya.iyer@example.com",
               "phone": "+91 98410 24680", "account_status": "active"},
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Earbuds", "amount": 3499, "status": "Delivered",
         "order_date": "2026-10-14", "delivered_date": "2026-10-18", "tracking": "AWB-7781229034"},
        {"order_id": "ORD-5488", "item": "USB-C Hub (7-in-1)", "amount": 2199, "status": "Shipped",
         "order_date": "2026-10-26", "estimated_delivery": "2026-10-31", "tracking": "AWB-7781229871"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 6999, "status": "Delayed",
         "order_date": "2026-10-15", "estimated_delivery": "2026-10-21", "tracking": "AWB-7781227745"},
    ],
    "C-1003": [
        {"order_id": "ORD-5301", "item": "Smartwatch", "amount": 12999, "status": "Delivered",
         "order_date": "2026-09-02", "delivered_date": "2026-09-06", "tracking": "AWB-7781221108"},
        {"order_id": "ORD-5602", "item": "Laptop (14-inch, 16 GB RAM)", "amount": 72990, "status": "Delivered",
         "order_date": "2026-10-20", "delivered_date": "2026-10-23", "tracking": "AWB-7781230456"},
    ],
}


def find_order(order_id: str) -> dict | None:
    for orders in ORDERS.values():
        for order in orders:
            if order["order_id"] == order_id:
                return order
    return None


def inr(amount: float) -> str:
    return f"₹{amount:,.0f}"


# --- Tools ---

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer's profile by their customer ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"No customer found with ID {customer_id}"
    return (f"Customer: {c['name']} ({c['city']})\nEmail: {c['email']}\n"
            f"Phone: {c['phone']}\nAccount status: {c['account_status']}")


@tool
def get_order_history(customer_id: str) -> str:
    """Get the order history for a customer, including status, dates and tracking.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for o in orders:
        line = f"{o['order_id']}: {o['item']} — {inr(o['amount'])} [{o['status']}] ordered {o['order_date']}"
        if o.get("delivered_date"):
            line += f", delivered {o['delivered_date']}"
        if o.get("estimated_delivery"):
            line += f", estimated delivery {o['estimated_delivery']}"
        lines.append(line + f", tracking {o['tracking']}")
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order to the customer's original payment method.

    Args:
        order_id: The order ID to refund (e.g. ORD-5521)
        amount: The refund amount in Indian Rupees
    """
    return (f"Refund REF-{order_id[4:]} of {inr(amount)} initiated for {order_id} to the original "
            "payment method (UPI / card / net banking). It will reflect in 5-7 business days.")


@tool
def create_support_ticket(customer_id: str, reason: str, priority: str = "normal") -> str:
    """Escalate an issue to a human support specialist by creating a ticket.

    Args:
        customer_id: The customer ID (e.g. C-1001)
        reason: Short description of why this needs a human
        priority: "normal" or "high"
    """
    return (f"Ticket TKT-{abs(hash((customer_id, reason))) % 90000 + 10000} created for {customer_id} "
            f"({priority} priority). A specialist will respond within 24 hours.")


ALL_TOOLS = [lookup_customer, get_order_history, process_refund, create_support_ticket]


import os
from datetime import date

# Swapping models is a config change: e.g. amazon.nova-pro-v1:0 or a Claude Haiku profile
MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")


def build_system_prompt() -> str:
    today = date.today().strftime("%d %B %Y (%A)")
    return f"""You are the customer support agent for ElectroBazaar, an online electronics store in India.
Today's date is {today}.

Be helpful, professional and concise. Use the available tools to look up customer
information and act on requests. All prices are in Indian Rupees (₹).

Guidelines:
- Always ask for the customer ID first if you don't have it.
- Verify the customer with lookup_customer before taking any action on their account.
- Use tool data to answer questions — never ask the customer for information you already have.
- If you have skills available, activate the one that matches the request and follow it.
- Remember preferences the customer shares (contact channel, refund method, language) and honour them.
- Be warm but efficient."""


from strands.vended_plugins.skills import Skill

refund_processing_skill = Skill.from_content("""
---
name: refund-processing
description: Step-by-step process for handling customer refund and return requests, including order verification, return window checks, supervisor review and refund issuance.
---

# Refund Processing

1. **Verify the customer** with `lookup_customer`.

2. **Find the order** with `get_order_history`. Never ask the customer for dates or amounts the system already has.

3. **Check the return window.** Orders can be refunded within 30 days of the `delivered` date. Compare it with today's date from your instructions.
   - If the window has closed, explain this kindly and offer to raise a `create_support_ticket` for a goodwill exception.

4. **Check the amount.** If the refund is ₹50,000 or more, do NOT call `process_refund`. Tell the customer a supervisor must approve it, create a `high` priority support ticket, and say they will hear back within 24 hours.

5. **Confirm with the customer.** State the order ID, item and refund amount in one line and ask for a yes/no.

6. **Process the refund** with `process_refund` only after the customer says yes.

7. **Set expectations.** Refunds go back to the original payment method (UPI / card / net banking) in 5-7 business days.

## Notes
- Be empathetic. The customer has a problem they want solved.
- Never refund more than the order value.
""")

order_tracking_skill = Skill.from_content("""
---
name: order-tracking
description: Help customers check the status of their orders, understand delivery timelines, and handle delayed or missing shipments.
---

# Order Tracking

1. **Verify the customer** with `lookup_customer`.

2. **Get the order** with `get_order_history` and explain the status:
   - **Processing**: ships within 1-2 business days.
   - **Shipped**: share the AWB tracking number and estimated delivery date.
   - **Delivered**: confirm the delivery date.
   - **Delayed**: follow the delay steps below.

3. **If the order is delayed** (today is past the estimated delivery date):
   - Acknowledge the frustration and apologise sincerely.
   - Offer a choice: keep waiting with a ₹200 store credit, or cancel for a full refund.
   - If they want to wait, create a `normal` priority support ticket to chase the courier.

## Notes
- Always share the tracking number when available.
- If an order shows "Delivered" but the customer hasn't received it, create a `high` priority ticket and promise a response within 24 hours.
""")

account_troubleshooting_skill = Skill.from_content("""
---
name: account-troubleshooting
description: Guide for helping customers with account issues including login problems, OTP issues, locked accounts and updating contact details.
---

# Account Troubleshooting

1. **Verify the customer** with `lookup_customer`. Confirm their name and registered email match before discussing account changes.

2. **Identify the issue:**
   - **Not receiving OTP**: check that the registered mobile number is correct and ask them to wait 60 seconds before retrying. If it still fails, create a support ticket.
   - **Locked account** (5 failed attempts): explain that it unlocks automatically after 30 minutes, or create a ticket for an immediate unlock.
   - **Update phone/email**: changes need verification on both the old and new contact. Create a support ticket to start the change.

## Notes
- Never reveal full contact details or payment information. Mask them (e.g. +91 98xxx xx345).
- If the customer can't verify basic details, politely escalate to the security team for their protection.
""")

SKILLS = [refund_processing_skill, order_tracking_skill, account_troubleshooting_skill]

In [ ]:
import time
from datetime import datetime

import boto3
from strands import Agent, AgentSkills
from strands.models import BedrockModel

REGION = boto3.session.Session().region_name or "us-east-1"
print(f"Region: {REGION}")

### The problem: a restart forgets everything

In [ ]:
agent = Agent(model=BedrockModel(model_id=MODEL_ID), tools=ALL_TOOLS, system_prompt=build_system_prompt(),
              callback_handler=None)
result = agent("Hi, I'm customer C-1003, Kavya.")

restarted = Agent(model=BedrockModel(model_id=MODEL_ID), tools=ALL_TOOLS, system_prompt=build_system_prompt())
result = restarted("What's my customer ID?")

### Step 1 — Create an AgentCore Memory resource (one-time)

The **strategies** tell AgentCore what to extract in the background:
- `userPreferenceMemoryStrategy` → how the customer likes to be served
- `semanticMemoryStrategy` → facts (orders discussed, issues raised)
- `summaryMemoryStrategy` → a rolling summary of each session

Takes about 2-3 minutes the first time; after that it's reused.

In [ ]:
import logging
from bedrock_agentcore.memory import MemoryClient

logging.getLogger("bedrock_agentcore.memory").setLevel(logging.CRITICAL)  # quiet the "already exists" log on reuse
memory_client = MemoryClient(region_name=REGION)

memory = memory_client.create_or_get_memory(
    name="ElectroBazaarSupportMemory",
    description="Short and long-term memory for the ElectroBazaar support agent",
    event_expiry_days=30,
    strategies=[
        {"userPreferenceMemoryStrategy": {"name": "CustomerPreferences",
                                          "namespaces": ["/preferences/{actorId}/"]}},
        {"semanticMemoryStrategy": {"name": "CustomerFacts",
                                    "namespaces": ["/facts/{actorId}/"]}},
        {"summaryMemoryStrategy": {"name": "SessionSummaries",
                                   "namespaces": ["/summaries/{actorId}/{sessionId}/"]}},
    ],
)
MEMORY_ID = memory["id"]

while (status := memory_client.get_memory_status(MEMORY_ID)) != "ACTIVE":
    print(f"Memory status: {status} … waiting")
    time.sleep(15)
print(f"✅ Memory ready: {MEMORY_ID}")

### Step 2 — Plug memory into the agent

`actor_id` = **who** (the customer). `session_id` = **which conversation**.
`retrieval_config` = which long-term namespaces to search and inject on each turn.

In [ ]:
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

ACTOR_ID = "C-1003"   # Kavya


def support_agent_with_memory(session_id: str) -> Agent:
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        actor_id=ACTOR_ID,
        session_id=session_id,
        retrieval_config={
            "/preferences/{actorId}/": RetrievalConfig(top_k=5, relevance_score=0.3),
            "/facts/{actorId}/": RetrievalConfig(top_k=5, relevance_score=0.3),
        },
    )
    return Agent(
        model=BedrockModel(model_id=MODEL_ID),
        tools=ALL_TOOLS,
        plugins=[AgentSkills(skills=SKILLS)],
        system_prompt=build_system_prompt(),
        session_manager=AgentCoreMemorySessionManager(config, region_name=REGION),
    )


SESSION_1 = f"kavya-chat-{datetime.now():%Y%m%d-%H%M%S}"
agent = support_agent_with_memory(SESSION_1)
result = agent("Hi, I'm Kavya, customer C-1003. Quick note for the future: please contact me only on WhatsApp, "
      "never phone calls, and I prefer store credit over bank refunds whenever possible.")

In [ ]:
result = agent("Can you check the status of my laptop order?")

### Step 3 — Short-term memory: survive a restart

Simulate a crash or redeploy: a **brand new agent object**, same `session_id`. The conversation is restored from AgentCore.

In [ ]:
del agent
agent = support_agent_with_memory(SESSION_1)
print(f"🔄 Restored {len(agent.messages)} messages from AgentCore Memory\n")
result = agent("Sorry, which order were we just talking about?")

### Step 4 — Long-term memory: what did AgentCore learn?

Extraction runs asynchronously in the background, usually within about a minute.

In [ ]:
import json


def show_records(namespace):
    records = memory_client.retrieve_memories(memory_id=MEMORY_ID, namespace=namespace,
                                              query="customer preferences and contact", top_k=5)
    for r in records:
        text = r.get("content", {}).get("text", "")
        try:
            text = json.loads(text).get("preference", text)  # preference records are JSON
        except (ValueError, AttributeError):
            pass
        print("  •", text)

for attempt in range(12):
    prefs = memory_client.retrieve_memories(memory_id=MEMORY_ID, namespace=f"/preferences/{ACTOR_ID}/",
                                            query="contact and refund preferences", top_k=5)
    if prefs:
        break
    print("⏳ waiting for long-term extraction…")
    time.sleep(15)

print("\nPreferences learned:")
show_records(f"/preferences/{ACTOR_ID}/")
print("\nFacts learned:")
show_records(f"/facts/{ACTOR_ID}/")

### Step 5 — Kavya comes back next week

A **new session** with no shared conversation history. The agent still knows how Kavya likes to be served.

In [ ]:
SESSION_2 = f"kavya-chat-{datetime.now():%Y%m%d-%H%M%S}-new"
new_visit = support_agent_with_memory(SESSION_2)
result = new_visit("Hi, it's Kavya again. If something goes wrong with an order, how will you reach me "
          "and how would a refund be given to me?")

---
### Takeaways

- `AgentCoreMemorySessionManager` adds durable memory with **zero changes** to tools, hooks or policy.
- **Short-term** memory makes the agent stateless-safe: any instance can resume any session.
- **Long-term** memory turns conversations into customer knowledge, extracted automatically.

All the pieces are ready. Time to ship.

Next: **05_deploy_to_agentcore.ipynb**